In [ ]:
import bz2
import re
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from wordcloud import WordCloud
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\skadi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\skadi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
reviews = [] # areates an empty list

file_path = "train.ft.txt.bz2"

with bz2.open(file_path, "rt", encoding="utf-8") as file:
    for i, line in enumerate(file): # Load first 50,000 reviews
        if i == 50000:
            break
        label, text = line.split(' ', 1)

        reviews.append([label, text])

df = pd.DataFrame(reviews, columns=["label", "review"])

print(df.head())
print("\nDataset Shape:", df.shape)


        label                                             review
0  __label__2  Stuning even for the non-gamer: This sound tra...
1  __label__2  The best soundtrack ever to anything.: I'm rea...
2  __label__2  Amazing!: This soundtrack is my favorite music...
3  __label__2  Excellent Soundtrack: I truly like this soundt...
4  __label__2  Remember, Pull Your Jaw Off The Floor After He...

Dataset Shape: (50000, 2)


In [ ]:

df["char_length"] = df["review"].apply(len) # create a new columan and stored number of char
df["word_count"] = df["review"].apply(lambda x: len(x.split())) # create a new columan and total num of each review 
empty_reviews = (df["review"].str.strip() == "").sum() # remove spaces 
duplicate_reviews = df["review"].duplicated().sum()

print("Total Reviews:", len(df))
print("Empty Reviews:", empty_reviews)
print("Duplicate Reviews:", duplicate_reviews)

print("\nAverage Character Length:", df["char_length"].mean())
print("Average Word Count:", df["word_count"].mean())

Total Reviews: 50000
Empty Reviews: 0
Duplicate Reviews: 0

Average Character Length: 442.0161
Average Word Count: 80.17758


In [ ]:
def clean_text(text):


    text = text.lower() # conver all word  in lowecase 
    text = re.sub(r'https?://\S+', ' ', text) # remove url 
    text = re.sub(r'<.*?>', ' ', text) # remove html word 
    text = re.sub(r'\S+@\S+', ' ', text) # removes email addresses.
    text = re.sub(r'\d+', ' ', text)# Remove punctuation/special chars
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces

    return text

In [ ]:
df["clean_review"] = df["review"].apply(clean_text) # applie every row in "review" column.
print(df[["review", "clean_review"]].head())

                                              review  \
0  Stuning even for the non-gamer: This sound tra...   
1  The best soundtrack ever to anything.: I'm rea...   
2  Amazing!: This soundtrack is my favorite music...   
3  Excellent Soundtrack: I truly like this soundt...   
4  Remember, Pull Your Jaw Off The Floor After He...   

                                        clean_review  
0  stuning even for the non gamer this sound trac...  
1  the best soundtrack ever to anything i m readi...  
2  amazing this soundtrack is my favorite music o...  
3  excellent soundtrack i truly like this soundtr...  
4  remember pull your jaw off the floor after hea...  


In [ ]:
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    tokens = word_tokenize(text)
    return tokens
df["tokens"] = df["clean_review"].apply(tokenize_text)
print(df["tokens"].head())

0    [stuning, even, for, the, non, gamer, this, so...
1    [the, best, soundtrack, ever, to, anything, i,...
2    [amazing, this, soundtrack, is, my, favorite, ...
3    [excellent, soundtrack, i, truly, like, this, ...
4    [remember, pull, your, jaw, off, the, floor, a...
Name: tokens, dtype: object


In [ ]:
audit_df = pd.DataFrame([
    [
        len(t), # counts total tokens in review
        sum(w in stop_words for w in t), # counts stopwords in token list
        len(t) - sum(w in stop_words for w in t), # calculates meaningful word
        sum(w in stop_words for w in t)/len(t) if len(t)>0 else 0, # calculate % of stopword
        (len(t)-sum(w in stop_words for w in t))/len(t) if len(t)>0 else 0 #  calculate % of meaninigfull word
    ]
    for t in df["tokens"]
], columns=[
    "total_tokens",
    "stopword_tokens",
    "informative_tokens",
    "stopword_ratio",
    "informative_ratio"
])

print(audit_df.head())

   total_tokens  stopword_tokens  informative_tokens  stopword_ratio  \
0            80               36                  44        0.450000   
1           102               56                  46        0.549020   
2           135               67                  68        0.496296   
3           122               47                  75        0.385246   
4            90               44                  46        0.488889   

   informative_ratio  
0           0.550000  
1           0.450980  
2           0.503704  
3           0.614754  
4           0.511111  


In [ ]:
bigram_counter = Counter(
    bg
    for tokens in df["tokens"]
    for bg in ngrams(
        [t for t in tokens if t not in stop_words and len(t) > 2],2 # remove stopwords and remove short words
    )
)
for bg, count in bigram_counter.most_common(20):
    print(bg, ":", count)

('read', 'book') : 2035
('one', 'best') : 1072
('waste', 'money') : 1028
('year', 'old') : 1021
('would', 'recommend') : 990
('waste', 'time') : 908
('great', 'book') : 883
('much', 'better') : 819
('highly', 'recommend') : 772
('years', 'ago') : 769
('book', 'read') : 746
('first', 'time') : 719
('good', 'book') : 710
('reading', 'book') : 696
('ever', 'read') : 659
('recommend', 'book') : 613
('even', 'though') : 609
('great', 'movie') : 563
('long', 'time') : 536
('well', 'written') : 531


In [13]:
trigram_counter = Counter(
    tg
    for tokens in df["tokens"]
    for tg in ngrams(
        [t for t in tokens if t not in stop_words and len(t) > 2], 3
    )
)

for tg, count in trigram_counter.most_common(20):
    print(tg, ":", count)

('book', 'ever', 'read') : 281
('waste', 'time', 'money') : 206
('books', 'ever', 'read') : 175
('would', 'recommend', 'book') : 164
('recommend', 'book', 'anyone') : 157
('would', 'recommend', 'anyone') : 154
('worst', 'movie', 'ever') : 142
('one', 'best', 'books') : 137
('worst', 'book', 'ever') : 125
('best', 'book', 'ever') : 124
('highly', 'recommend', 'book') : 123
('year', 'old', 'son') : 116
('movie', 'ever', 'seen') : 116
('would', 'highly', 'recommend') : 110
('pampers', 'baby', 'dry') : 106
('best', 'books', 'ever') : 92
('one', 'worst', 'movies') : 88
('year', 'old', 'daughter') : 86
('movies', 'ever', 'seen') : 82
('movie', 'ever', 'made') : 78


In [14]:
def detect_issues(text):
    issues = []

    if re.search(r'https?://\S+', text): issues.append("URL")
    if re.search(r'\S+@\S+', text): issues.append("EMAIL")
    if re.search(r'<.*?>', text): issues.append("HTML")
    if len(text.split()) < 3: issues.append("TOO_SHORT")
    if len(text.split()) > 300: issues.append("TOO_LONG")

    return ", ".join(issues)

df["quality_issues"] = df["review"].apply(detect_issues)

print(df[["review", "quality_issues"]].head())

                                              review quality_issues
0  Stuning even for the non-gamer: This sound tra...               
1  The best soundtrack ever to anything.: I'm rea...               
2  Amazing!: This soundtrack is my favorite music...               
3  Excellent Soundtrack: I truly like this soundt...               
4  Remember, Pull Your Jaw Off The Floor After He...               


In [17]:
df.to_csv("cleaned_reviews.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
